# **<span style="color: black; font-size:1em;">IDAES-LCA Integration (ILI)</span>**

## <span style="color:black; font-weight:bold"> Introduction </span> 

<span style = "color:black">
This document demonstrates the integration of Life Cycle Assessment into the IDAES workflow. The goal is to develop a solution allowing a bi-directional communication between PrOMMiS and openLCA, and enabling a selective, constraint-based, environmental optimization framework 
</br>

<u> Goal and Scope:</u> In this Jupyter Notebook, we use life cycle assessment to evaluate the environmental impact of Rare Earth Oxide Recovery from coal mining refuse.</br>
As shown in the system boundary below, the scope of this work starts with the leaching of size-reduced REE-rich feedstock (REE: Rare Earth Elements) and ends with the recovery of mixed REO solids. The process consists of six main stages: 1) Mixing and Leaching, 2) Rougher Solvent Extraction, 3) Cleaner Solvent Extraction, 4) Precipitation, 5) Solid-Liquid (S/L) separation, and 6) Roasting. </br>
It's crucial to note here that the current script does not account for:

* Upstream processes leading to the production of REE-rich feedstock 


* Downstream processes leading to the separation of REE contained in the REO    

<u> Main Product:</u> The main product of the flowsheet modeled in this Jupyter Notebook is Rare Earth Oxide (REO) solid.

<u> Co-products or by-products:</u> None

<u> Functional Unit:</u> The function unit is 1 kg of recovered REO solids

<u> Allocation:</u> The evaluated flowsheet produces a single product with no intermediate co-products or by-product. As a result, there is no need for allocation in this modeling exercise.

<u> Life cycle inventory Estimation:</u> The material and energy inputs shown in the system boundary figure below have been shortlisted and estimated based on the UKy flowsheet output, [technical reports on the production on the production of REE from coal and coal by-products](https://www.osti.gov/servlets/purl/1569277), and other relevant literature.
</span>

<span style = "color:black">

**Useful notes, definitions, and links:**
- **PrOMMiS model:** Process Optimization and Modeling for Minerals Sustainability initiative, led by the U.S. Department of Energy (DOE), specifically under NETL (National Energy Technology Laboratory). The PrOMMiS source code can be accessed on github through the following repository link: [https://github.com/prommis/prommis/tree/main](https://github.com/prommis/prommis/tree/main)<br> PrOMMiS serves to optimize processing flowsheets. This Jupyter Notebook focuses on a flowsheet for the extraction of REE from coal refuse. 


- **UKy Flowsheet Code:** [https://github.com/prommis/prommis/blob/main/src/prommis/uky/uky_flowsheet.py](https://github.com/prommis/prommis/blob/main/src/prommis/uky/uky_flowsheet.py)

</span>

<div style="text-align: center;">
    <img src="images/system_boundary_1.png" width="1000"/>
</div>

## <span style="color:black; font-weight:bold"> Step 1: Import the necessary tools and setup output directory</span> 

In [ ]:
import src.foqus_class as foqus_class
from src.foqus_class import NetlFoqus
from src.foqus_class import openlca_node_script, prommis_node_script
import os
import logging
from pathlib import Path
import pandas as pd
import olca_schema as olca
from netlolca.NetlOlca import NetlOlca
import prommis.uky.uky_flowsheet as uky
from prommis.uky.costing.ree_plant_capcost import QGESSCostingData
from pyomo.environ import TransformationFactory
from pyomo.core.base.var import Var
from idaes.core.util.model_diagnostics import DiagnosticsToolbox
import foqus_lib.framework.graph.graph as gr
import foqus_lib.framework.graph.node as nd
import foqus_lib.framework.graph.edge as ed
import foqus_lib.framework.graph.nodeVars as nv
from foqus_lib.framework.uq.Distribution import Distribution
from foqus_lib.framework.session.session import session
from foqus_lib.framework.optimizer.problem import objectiveFunction
import src as lca_prommis
import foqus_lib.framework.optimizer.NLopt as nlopt
from foqus_lib.framework.optimizer.problem import inequalityConstraint

output_dir = Path.home() / "output" 
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

## <span style="color:black; font-weight:bold"> Step 2: Create a New FOQUS Graph and initiate UKy flowsheet  </span> 

In [ ]:
nf = NetlFoqus()
m = nf.init_uky()

## <span style="color:black; font-weight:bold"> Step 3: FOQUS - Setup the PrOMMiS and openLCA Nodes </span> 

#### <span style="color:black; font-weight:bold"> Select the Decision Variables </span> 

In [ ]:
# Add a decision variable
my_var1 = "fs.leach_liquid_feed.flow_vol"
nf.add_decision_variable(my_var1)

# [FH] we need at least two decision variables
my_var2 = "fs.load_sep.split_fraction"
nf.add_decision_variable(my_var2)

# Help with initializing decision variables
foqus_class.initialize_decision_variables(nf, m)

# Store decision variable information in a dataframe and save to output directory
dv_data = []
for dv in nf.dv:
    dv_data.append({
        'variable_name': dv.ipvname,
        'value': dv.value,
        'min': dv.min,
        'max': dv.max,
        'scaling': dv.scaling
    })

dv_df = pd.DataFrame(dv_data)
dv_df.to_csv(output_dir / "decision_variables.csv", index=False)

# set decesion variables as input variables for prommis_node
for dv in nf.dv:
    nf.set_input_variables(nf.prommis_node, dv.ipvname, dv.value, dv.min, dv.max)


#### <span style="color:black; font-weight:bold"> Initialize the intermediate variables </span> 

In [ ]:
# Initialize intermediate variables
nf.initialize_intermediate_variables(nf.prommis_node, nf.olca_node)

# connect intermediate variables
nf.connect_intermediate_variables(nf.prommis_node, nf.olca_node)

# save exchanges in nf.exchanges
lca_df_finalized = nf.exchanges

#### <span style="color:black; font-weight:bold"> Initialize the LCA model </span> 

##### <u><span style="color:black; font-weight:bold">Connect to IPC</span></u>

In [ ]:
netl = NetlOlca()
netl.connect()
netl.read()

##### <u><span style="color:black; font-weight:bold">Enter Process Name and Description</span></u>

In [ ]:
process_name = "TESTING - REO Extraction From Coal Mining Refuse | UKy Flowsheet"

process_description = """This process involves the production of a Rare 
Earth Oxide solid extraction from coal mining refuse. The scope of this 
work starts with the leaching of size-reduced REE-rich feedstock (REE: 
Rare Earth Elements) and ends with the recovery of mixed REO solids. The 
process consists of six main stages: 1) Mixing and Leaching, 2) Rougher 
Solvent Extraction, 3) Cleaner Solvent Extraction, 4) Precipitation, 
5) Solid-Liquid (S/L) separation, and 6) Roasting. This process does not 
account for upstream processes leading to the production of REE-rich 
feedstock nor does it account for Downstream processes leading to the 
separation of REE contained in the REO. The main product is a rare earth 
oxide solid with no other by-products or co-products. The functional 
unit is 1 kg of recovered REO solids. The material and energy inputs 
shown in the system boundary figure below have been shortlisted and 
estimated based on the UKy flowsheet output, as well as other relevant 
literature."""

##### <u><span style="color:black; font-weight:bold">Select Impact Assessment Method</span></u>

In [ ]:
impact_method_uuid = '60cb71ff-0ef0-4e6c-9ce7-c885d921dd15'

##### <u><span style="color:black; font-weight:bold">Configure Parameter Set</span></u>

In [ ]:
parameter_set_name = "Baseline"
parameter_set_description = "Baseline parameter set for the process"
is_baseline = True

##### <u><span style="color:black; font-weight:bold">Run first step in openLCA</span></u>

In [ ]:
total_impacts, my_parameters, ps_uuid = 
foqus_class.initiate_lca_model (netl, 
                                process_name, 
                                process_description, 
                                lca_df_finalized, 
                                impact_method_uuid, 
                                parameter_set_name, 
                                parameter_set_description, 
                                is_baseline)

##### <u><span style="color:black; font-weight:bold">Save model information</span></u>

In [ ]:
# create new df/file to store run info
run_info = pd.DataFrame(columns=['item', 'description'])
run_info.loc[len(run_info)] = ['ps_uuid', ps_uuid]
run_info.loc[len(run_info)] = ['impact_method_uuid', impact_method_uuid]
run_info.loc[len(run_info)] = ['parameter_set_name', parameter_set_name]
run_info.to_csv(output_dir / "run_info.csv", index=False)

# create new df/file to store results
total_impacts.to_csv(output_dir / "total_impacts.csv", index=False)

# save my_parameters to the output directory
my_parameters.to_csv(output_dir / "my_parameters.csv", index=False)

##### <u><span style="color:black; font-weight:bold">Create output variables of openLCA node in FOQUS</span></u>

In [ ]:
for impact_category in total_impacts['name']:
    nf.initiate_output_variables(nf.olca_node, 
                                impact_category, 
                                total_impacts.loc[total_impacts['name'] == impact_category, 'amount'].values[0])
    logging.info(
        "Initiated output variable %s for node %s: amount=%f" % (
        impact_category, 
        nf.olca_node.name, 
        total_impacts.loc[total_impacts['name'] == impact_category, 'amount'].values[0]
        )
    )

##### <u><span style="color:black; font-weight:bold">Define openLCA and PrOMMiS node scripts</span></u>

In [ ]:
#openLCA node
nf.define_node_script(nf.olca_node, openlca_node_script)

#PrOMMiS node
nf.define_node_script(nf.prommis_node, prommis_node_script)

## <span style="color:black; font-weight:bold"> Step 4: Create Session </span> 

In [ ]:
# Enter your foqus working directory
foqus_wd = ""
my_session = nf.create_session(foqus_wd)

## <span style="color:black; font-weight:bold"> Step 5: Create a problem and setup optimizer </span> 

In [ ]:
problem = nf.setup_optimizer(my_session, "NLopt", nf.prommis_node) # first step in setting up optimizer


## <span style="color:black; font-weight:bold"> Step 6: Create Problem Objective </span> 

In [ ]:
problem = nf.create_problem_objective(problem, 
                                    ["Global Warming Potential [AR6, 100 yr]"], 
                                    nf.olca_node) # create problem objective

## <span style="color:black; font-weight:bold"> Step 7: Setup solver options </span> 

In [ ]:
problem = nf.setup_nlopt_solver_options(problem, True) 

## <span style="color:black; font-weight:bold"> Step 8: Run Optimization </span> 

In [ ]:
my_solver, problem = nf.run_optimization(problem, my_session) 